# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, record sets define tables or collections of records. Each record set, field, and column is uniquely identified by its `@id`.

Let's enumerate all available record sets and the main fields in this dataset.

In [ ]:
# Retrieve all record sets in the dataset
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet name: {rs.name} | @id: {rs.id}")

# Display all fields (columns) for each record set
for rs in record_sets:
    print(f"---\nRecordSet: {rs.name} (ID: {rs.id})")
    for field in rs.fields:
        print(f"  Field name: {field.name} | @id: {field.id} | Data Type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we demonstrate how to extract all data tables in the dataset using their `@id`s. Data is loaded into Pandas DataFrames for further processing.

In [ ]:
dataframes = dict()
recordset_ids = [rs.id for rs in record_sets]

# Load each record set as a DataFrame
for rs_id in recordset_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# List the columns for each DataFrame
for rs_id in recordset_ids:
    print(f"RecordSet @id: {rs_id}")
    print("Columns:", dataframes[rs_id].columns.tolist())
    print()

For subsequent analysis, select the main data record set (usually the table with the main measurements). For this dataset, let's pick the first record set listed above.

In [ ]:
# Pick the first record set for further EDA
main_rs_id = recordset_ids[0]
main_df = dataframes[main_rs_id]
print(f"Selected main record set: {main_rs_id}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Filter on a numeric field (`Peptide_count` for proteins for example)
- Normalize this field
- Group by a categorical field (e.g. `Description`)

First, let's list all numeric and categorical fields.

In [ ]:
# Identify numeric fields in the main DataFrame
numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
print("Numeric fields:", numeric_fields)

# Identify likely categorical fields (object but not free-form text)
categorical_fields = [col for col in main_df.columns if main_df[col].dtype=='object']
print("Categorical fields:", categorical_fields)

# For this example, pick the first numeric and categorical fields
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    raise ValueError("No numeric field found.")

if categorical_fields:
    group_field = categorical_fields[0]
else:
    raise ValueError("No categorical field found.")

print(f"Using numeric field: {numeric_field}")
print(f"Using categorical field: {group_field}")

In [ ]:
# Remove records with missing or invalid numeric values
filtered_df = main_df[pd.to_numeric(main_df[numeric_field], errors='coerce').notnull()].copy()
filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field])

# Filter records where the field value > threshold
threshold = filtered_df[numeric_field].mean()
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by categorical field and compute mean
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
    print(f"\nGrouped data by {group_field} (mean {numeric_field} per group):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the chosen numeric field and display the mean values by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there are few groups, show mean by group
if group_field in filtered_df.columns:
    group_counts = filtered_df[group_field].value_counts()
    top_groups = group_counts.head(10).index
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=filtered_df[filtered_df[group_field].isin(top_groups)],
        x=group_field, y=numeric_field
    )
    plt.title(f"Mean {numeric_field} by Top Groups in {group_field}")
    plt.xlabel(group_field)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load the FAIR<sup>2</sup> mass spectrometry dataset, explored record structures using their Croissant `@id`s, performed filtering and normalization on a numeric field, grouped by a categorical attribute, and visualized our analysis. This workflow demonstrates a reproducible approach to interacting with semantic datasets using unique identifiers for robust, FAIR data science.